<a href="https://colab.research.google.com/github/MatthewAlvarez5/CSCI164_ProblemSolving/blob/main/AI25Ch3a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tile Sliding Domain: Functions

In [1]:
import random
import heapq
import time
from collections import deque

### General Settings

# This will be overridden as needed
StateDimension = 3


In [2]:
### State Helpers

def Actions(s):
    return ['u', 'd', 'l', 'r']

def Opposite(action):
    return dict(u='d', d='u', l='r', r='l').get(action, None)

In [3]:
def Result(state, action):
    i = state.index(0)
    newState = list(state)
    row, col = divmod(i, StateDimension)

    if (action == 'u' and row == 0) or \
       (action == 'd' and row == StateDimension - 1) or \
       (action == 'l' and col == 0) or \
       (action == 'r' and col == StateDimension - 1):
        return newState

    if action == 'u':
        j = (row - 1) * StateDimension + col
    elif action == 'd':
        j = (row + 1) * StateDimension + col
    elif action == 'l':
        j = row * StateDimension + (col - 1)
    elif action == 'r':
        j = row * StateDimension + (col + 1)

    newState[i], newState[j] = newState[j], newState[i]
    return newState

In [4]:
def LegalMove(state, action):
    i = state.index(0)
    row, col = divmod(i, StateDimension)
    if (action == 'u' and row == 0) or \
       (action == 'd' and row == StateDimension - 1) or \
       (action == 'l' and col == 0) or \
       (action == 'r' and col == StateDimension - 1):
        return False
    return True

def PrintState(s):
    for i in range(0, len(s), StateDimension):
        print(s[i:i+StateDimension])

# RandomWalk

In [5]:
def RandomWalk(state, steps):
    actionSequence = []
    actionLast = None
    for _ in range(steps):
        action = None
        while action is None:
            a = random.choice(Actions(state))
            if LegalMove(state, a) and a != Opposite(actionLast):
                action = a
        actionLast = action
        state = Result(state, action)
        actionSequence.append(action)
    return state, actionSequence

In [6]:
def SingleTileManhattanDistance(tile, left, right):
    leftIndex = left.index(tile)
    rightIndex = right.index(tile)
    return abs(leftIndex // StateDimension - rightIndex // StateDimension) + \
           abs(leftIndex % StateDimension - rightIndex % StateDimension)

def ManhattanDistance(state, goal):
    return sum(SingleTileManhattanDistance(t, state, goal)
               for t in range(1, StateDimension ** 2))

In [7]:
def OutOfPlace(state, goal):
    return sum(1 for i in range(len(state)) if state[i] != goal[i] and state[i] != 0)


# Search Algorithms

In [8]:
### Search Algorithms

def BFS(start, goal):
    frontier = deque([(start, [])])
    explored = set()
    nodes = 0
    while frontier:
        state, path = frontier.popleft()
        nodes += 1
        if state == goal:
            return path, nodes
        explored.add(tuple(state))
        for a in Actions(state):
            if LegalMove(state, a):
                newState = Result(state, a)
                if tuple(newState) not in explored:
                    frontier.append((newState, path + [a]))
    return None, nodes

In [9]:
def AStar(start, goal, heuristic):
    frontier = []
    heapq.heappush(frontier, (heuristic(start, goal), 0, start, []))
    explored = set()
    nodes = 0
    while frontier:
        f, cost, state, path = heapq.heappop(frontier)
        nodes += 1
        if state == goal:
            return path, nodes
        explored.add(tuple(state))
        for a in Actions(state):
            if LegalMove(state, a):
                newState = Result(state, a)
                if tuple(newState) not in explored:
                    g = cost + 1
                    h = heuristic(newState, goal)
                    heapq.heappush(frontier, (g + h, g, newState, path + [a]))
    return None, nodes


# Test

In [10]:
### Experiment Runner

def RunTests(dim):
    ## intitial state space
    global StateDimension
    StateDimension = dim
    size = dim * dim
    initial = list(range(1, size)) + [0]
    goal = initial[:]
    depths = [5, 10, 20, 40, 80]
    all_results = []

    for depth in depths:
        for _ in range(3):
            s, _ = RandomWalk(initial[:], depth)
            bpath, bnodes = BFS(s, goal)
            apath1, anodes1 = AStar(s, goal, OutOfPlace)
            apath2, anodes2 = AStar(s, goal, ManhattanDistance)

            result = {
                'dimension': dim,
                'depth': depth,
                'start': s,
                'BFS': {'length': len(bpath), 'nodes': bnodes},
                'A*_OutOfPlace': {'length': len(apath1), 'nodes': anodes1},
                'A*_Manhattan': {'length': len(apath2), 'nodes': anodes2}
            }
            all_results.append(result)
            print(f"Dim {dim} | Depth {depth} | BFS len={len(bpath)} nodes={bnodes} | A*_OOP len={len(apath1)} nodes={anodes1} | A*_Manh len={len(apath2)} nodes={anodes2}")
    return all_results

In the following code, BFS is an uninformed search which blindly explores all possible states at each level. A* is informed search using heuristics to estimate the goal with more efficiently. As the depth increases, BFS expands exponentially more nodes than A*. The Manhattan heuristic outperforms Out-of-Place because it gives a more accurate cost estimate. A good heuristic drastically reduces the number of nodes expanded and makes deeper problems solvable. For future uses in AI and as the complexity increases, informed searches need to be paired with good heuristics as a tool.

In [ ]:
### Run All
if __name__ == '__main__':
    print("Running for 3x3 Puzzle...")
    results_3x3 = RunTests(3)
    print("\nRunning for 4x4 Puzzle...")
    results_4x4 = RunTests(4)


Running for 3x3 Puzzle...
Dim 3 | Depth 5 | BFS len=5 nodes=59 | A*_OOP len=5 nodes=6 | A*_Manh len=5 nodes=6
Dim 3 | Depth 5 | BFS len=5 nodes=52 | A*_OOP len=5 nodes=8 | A*_Manh len=5 nodes=7
Dim 3 | Depth 5 | BFS len=5 nodes=61 | A*_OOP len=5 nodes=6 | A*_Manh len=5 nodes=6
Dim 3 | Depth 10 | BFS len=10 nodes=724 | A*_OOP len=10 nodes=31 | A*_Manh len=10 nodes=11
Dim 3 | Depth 10 | BFS len=10 nodes=646 | A*_OOP len=10 nodes=43 | A*_Manh len=10 nodes=15
Dim 3 | Depth 10 | BFS len=10 nodes=734 | A*_OOP len=10 nodes=42 | A*_Manh len=10 nodes=18
Dim 3 | Depth 20 | BFS len=20 nodes=81671 | A*_OOP len=20 nodes=4831 | A*_Manh len=20 nodes=543
Dim 3 | Depth 20 | BFS len=18 nodes=34077 | A*_OOP len=18 nodes=1723 | A*_Manh len=18 nodes=442
Dim 3 | Depth 20 | BFS len=14 nodes=5650 | A*_OOP len=14 nodes=272 | A*_Manh len=14 nodes=48
Dim 3 | Depth 40 | BFS len=24 nodes=283748 | A*_OOP len=24 nodes=23998 | A*_Manh len=24 nodes=2939
Dim 3 | Depth 40 | BFS len=20 nodes=56045 | A*_OOP len=20 nodes=3

The results of the previous tests prior to the timeout:

Running for 3x3 Puzzle...

Dim 3 | Depth 5 | BFS len=5 nodes=59 | A*_OOP len=5 nodes=6 | A*_Manh len=5 nodes=6

Dim 3 | Depth 5 | BFS len=5 nodes=52 | A*_OOP len=5 nodes=8 | A*_Manh len=5 nodes=7

Dim 3 | Depth 5 | BFS len=5 nodes=61 | A*_OOP len=5 nodes=6 | A*_Manh len=5 nodes=6

Dim 3 | Depth 10 | BFS len=10 nodes=724 | A*_OOP len=10 nodes=31 | A*_Manh len=10 nodes=11

Dim 3 | Depth 10 | BFS len=10 nodes=646 | A*_OOP len=10 nodes=43 | A*_Manh len=10 nodes=15

Dim 3 | Depth 10 | BFS len=10 nodes=734 | A*_OOP len=10 nodes=42 | A*_Manh len=10 nodes=18

Dim 3 | Depth 20 | BFS len=20 nodes=81671 | A*_OOP len=20 nodes=4831 | A*_Manh len=20 nodes=543

Dim 3 | Depth 20 | BFS len=18 nodes=34077 | A*_OOP len=18 nodes=1723 | A*_Manh len=18 nodes=442

Dim 3 | Depth 20 | BFS len=14 nodes=5650 | A*_OOP len=14 nodes=272 | A*_Manh len=14 nodes=48

Dim 3 | Depth 40 | BFS len=24 nodes=283748 | A*_OOP len=24 nodes=23998 | A*_Manh len=24 nodes=2939

Dim 3 | Depth 40 | BFS len=20 nodes=56045 | A*_OOP len=20 nodes=3821 | A*_Manh len=20 nodes=629

Dim 3 | Depth 40 | BFS len=26 nodes=410723 | A*_OOP len=26 nodes=68613 | A*_Manh len=26 nodes=7949

Dim 3 | Depth 80 | BFS len=26 nodes=359226 | A*_OOP len=26 nodes=56585 | A*_Manh len=26 nodes=7067

Dim 3 | Depth 80 | BFS len=20 nodes=77973 | A*_OOP len=20 nodes=3486 | A*_Manh len=20 nodes=229

Dim 3 | Depth 80 | BFS len=22 nodes=156925 | A*_OOP len=22 nodes=8929 | A*_Manh len=22 nodes=806


Running for 4x4 Puzzle...

Dim 4 | Depth 5 | BFS len=5 nodes=84 | A*_OOP len=5 nodes=6 | A*_Manh len=5 nodes=6

Dim 4 | Depth 5 | BFS len=5 nodes=130 | A*_OOP len=5 nodes=6 | A*_Manh len=5 nodes=6

Dim 4 | Depth 5 | BFS len=5 nodes=118 | A*_OOP len=5 nodes=6 | A*_Manh len=5 nodes=6

Dim 4 | Depth 10 | BFS len=10 nodes=3731 | A*_OOP len=10 nodes=25 | A*_Manh len=10 nodes=15

Dim 4 | Depth 10 | BFS len=8 nodes=1246 | A*_OOP len=8 nodes=19 | A*_Manh len=8 nodes=11

Dim 4 | Depth 10 | BFS len=10 nodes=3210 | A*_OOP len=10 nodes=45 | A*_Manh len=10 nodes=18

Dim 4 | Depth 20 | BFS len=16 nodes=317585 | A*_OOP len=16 nodes=224 | A*_Manh len=16 nodes=64

Dim 4 | Depth 20 | BFS len=20 nodes=5295837 | A*_OOP len=20 nodes=2508 | A*_Manh len=20 nodes=83

Dim 4 | Depth 20 | BFS len=20 nodes=3957722 | A*_OOP len=20 nodes=2044 | A*_Manh len=20 nodes=53

The test ended because of how inefficient the BFS algorithm became with millions of nodes. A* is much more efficient in both types of heuristics.